# Hyper-parameter search

fANOVA importance of each parameter, and every trial's success against k.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

import utils as u

data_path = Path("../results")
figs_path = Path("figs")

In [ ]:
u.load_data(data_path)
figs_path.mkdir(exist_ok=True)

In [ ]:
OPTUNA_NAMES = {"k": "k (neighbours)", "one_per_episode": "≤ 1 neighbour per demo", "chunk_size": "chunk length",
                "use_consistency": "trajectory stickiness", "w_wrist": "wrist-camera weight",
                "temperature": "softmax temperature", "temporal_ensemble": "temporal ensembling",
                "distance_normalization": "distance normalisation", "w_agentview": "agent-view weight",
                "norm_preserving": "norm preserving", "use_history": "history matching", "use_action_mode": "mode seeking",
                "task_conditioning": "prompt conditioning", "state_weight": "state weight", "time_weight": "time weight",
                "use_time": "time feature", "memory_stride": "memory stride", "image_encoder": "image encoder",
                "pooling_resnet18": "feature pooling", "ensemble_coeff": "ensembling decay",
                "use_ensemble_mode": "ensemble mode seeking", "gripper_mode": "gripper averaging"}  # fmt: skip

In [ ]:
op = u.DATA["optuna"]
imp = list(op["importance"].items())[:8][::-1]
trials = [t for t in op["trials"] if "k" in t["params"]]

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(u.W, 3.1), layout="constrained", gridspec_kw={"width_ratios": [1.15, 1]})
a.barh(range(len(imp)), [100 * v for _k, v in imp], color=u.KNN, height=0.62, edgecolor="white")
a.set_yticks(range(len(imp)), [OPTUNA_NAMES.get(k, k) for k, _v in imp], fontsize=u.FS_SMALL - 2)
for i, (_k, v) in enumerate(imp):
    a.text(100 * v + 1, i, f"{100 * v:.0f}%", va="center", fontsize=u.FS_SMALL - 2.5, color=u.INK2)
a.set_xlabel("importance (fANOVA)", fontsize=u.FS_SMALL - 1)
a.set_xlim(0, 75)
u.hbar_axes(a)
for one, colour, lab in ((True, u.KNN, "≤ 1 per demo"), (False, u.HUMAN, "any frames")):
    sel = [t for t in trials if bool(t["params"].get("one_per_episode", True)) == one]
    b.scatter([t["params"]["k"] for t in sel], [100 * t["value"] for t in sel], s=34, color=colour,
              edgecolor="white", linewidth=0.8, label=lab, zorder=3)  # fmt: skip
b.set_xlabel("k", fontsize=u.FS_SMALL - 1)
b.set_ylabel("trial success (%)", fontsize=u.FS_SMALL - 1)
b.grid(True)
b.set_axisbelow(True)
b.legend(frameon=False, fontsize=u.FS_SMALL - 2.5, loc="lower right")
fig.savefig(figs_path / "optuna.pdf")